In [1]:
# ============================================================
# INSTALL & IMPORT LIBRARY
# ============================================================
!pip install ipywidgets -q

import re
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ============================================================
# DEFINISI TOKEN
# ============================================================

RESERVE_WORDS = {
    'if', 'else', 'elif', 'while', 'for', 'do', 'switch', 'case',
    'break', 'continue', 'return', 'function', 'func', 'def', 'class',
    'int', 'float', 'double', 'char', 'string', 'bool', 'boolean',
    'void', 'null', 'None', 'True', 'False', 'true', 'false',
    'import', 'from', 'as', 'in', 'not', 'and', 'or', 'is',
    'try', 'except', 'finally', 'raise', 'with', 'pass', 'lambda',
    'new', 'delete', 'this', 'self', 'super', 'static', 'public',
    'private', 'protected', 'const', 'var', 'let', 'print', 'input',
    'include', 'using', 'namespace', 'struct', 'enum', 'typedef',
    'sizeof', 'extern', 'volatile', 'register', 'goto', 'default',
    'cout', 'cin', 'endl', 'printf', 'scanf', 'malloc', 'free',
    'long', 'short', 'unsigned', 'signed', 'auto', 'inline'
}

SYMBOLS = {
    '(': 'LEFT_PAREN',
    ')': 'RIGHT_PAREN',
    '{': 'LEFT_BRACE',
    '}': 'RIGHT_BRACE',
    '[': 'LEFT_BRACKET',
    ']': 'RIGHT_BRACKET',
    ';': 'SEMICOLON',
    ':': 'COLON',
    ',': 'COMMA',
    '.': 'DOT',
    '#': 'HASH',
    '@': 'AT',
    '~': 'TILDE',
    '?': 'QUESTION_MARK',
    '\\': 'BACKSLASH',
}

MATH_OPERATORS = {
    '+': 'PLUS',
    '-': 'MINUS',
    '*': 'MULTIPLY',
    '/': 'DIVIDE',
    '%': 'MODULO',
    '^': 'POWER',
    '=': 'ASSIGN',
    '<': 'LESS_THAN',
    '>': 'GREATER_THAN',
    '==': 'EQUAL',
    '!=': 'NOT_EQUAL',
    '<=': 'LESS_EQUAL',
    '>=': 'GREATER_EQUAL',
    '+=': 'PLUS_ASSIGN',
    '-=': 'MINUS_ASSIGN',
    '*=': 'MULTIPLY_ASSIGN',
    '/=': 'DIVIDE_ASSIGN',
    '++': 'INCREMENT',
    '--': 'DECREMENT',
    '**': 'POWER',
    '//': 'FLOOR_DIVIDE',
    '&&': 'LOGICAL_AND',
    '||': 'LOGICAL_OR',
    '<<': 'LEFT_SHIFT',
    '>>': 'RIGHT_SHIFT',
    '->': 'ARROW',
    '::': 'SCOPE',
}

# ============================================================
# FUNGSI TOKENIZER
# ============================================================

def tokenize(source_code):
    tokens = {
        'reserve_words': [],
        'symbols': [],
        'variables': [],
        'math_expressions': [],
        'strings': [],
        'comments': [],
        'unknown': []
    }

    all_tokens = []

    token_pattern = re.compile(
        r'(?P<COMMENT_ML>/\*[\s\S]*?\*/)'
        r'|(?P<COMMENT_SL>//[^\n]*)'
        r'|(?P<COMMENT_PY>#[^\n]*)'
        r'|(?P<STRING_D>"(?:[^"\\]|\\.)*")'
        r'|(?P<STRING_S>\'(?:[^\'\\]|\\.)*\')'
        r'|(?P<FLOAT>\d+\.\d+)'
        r'|(?P<INTEGER>\d+)'
        r'|(?P<OP_DOUBLE>[=!<>+\-*/&|:]{2})'
        r'|(?P<IDENTIFIER>[a-zA-Z_][a-zA-Z0-9_]*)'
        r'|(?P<OPERATOR>[+\-*/%^=<>!&|])'
        r'|(?P<SYMBOL>[(){}\[\];:,\.#@~?\\])'
        r'|(?P<NEWLINE>\n)'
        r'|(?P<WHITESPACE>[ \t]+)'
        r'|(?P<UNKNOWN>.)'
    )

    line_number = 1

    for match in token_pattern.finditer(source_code):
        kind = match.lastgroup
        value = match.group()

        if kind == 'NEWLINE':
            line_number += 1
            continue
        elif kind == 'WHITESPACE':
            continue
        elif kind in ('COMMENT_ML', 'COMMENT_SL', 'COMMENT_PY'):
            token_info = {
                'token': value.strip(),
                'type': 'Komentar',
                'subtype': kind,
                'line': line_number
            }
            tokens['comments'].append(token_info)
            all_tokens.append(token_info)
        elif kind in ('STRING_D', 'STRING_S'):
            token_info = {
                'token': value,
                'type': 'String Literal',
                'subtype': kind,
                'line': line_number
            }
            tokens['strings'].append(token_info)
            all_tokens.append(token_info)
        elif kind in ('FLOAT', 'INTEGER'):
            token_info = {
                'token': value,
                'type': 'Kalimat Matematika',
                'subtype': f'Number ({kind})',
                'line': line_number
            }
            tokens['math_expressions'].append(token_info)
            all_tokens.append(token_info)
        elif kind == 'IDENTIFIER':
            if value in RESERVE_WORDS:
                token_info = {
                    'token': value,
                    'type': 'Reserve Word',
                    'subtype': f'KEYWORD',
                    'line': line_number
                }
                tokens['reserve_words'].append(token_info)
            else:
                token_info = {
                    'token': value,
                    'type': 'Variabel',
                    'subtype': 'IDENTIFIER',
                    'line': line_number
                }
                tokens['variables'].append(token_info)
            all_tokens.append(token_info)
        elif kind == 'OP_DOUBLE':
            op_name = MATH_OPERATORS.get(value, f'OP_{value}')
            token_info = {
                'token': value,
                'type': 'Kalimat Matematika',
                'subtype': f'Operator ({op_name})',
                'line': line_number
            }
            tokens['math_expressions'].append(token_info)
            all_tokens.append(token_info)
        elif kind == 'OPERATOR':
            op_name = MATH_OPERATORS.get(value, f'OP_{value}')
            token_info = {
                'token': value,
                'type': 'Kalimat Matematika',
                'subtype': f'Operator ({op_name})',
                'line': line_number
            }
            tokens['math_expressions'].append(token_info)
            all_tokens.append(token_info)
        elif kind == 'SYMBOL':
            sym_name = SYMBOLS.get(value, f'SYMBOL_{value}')
            token_info = {
                'token': value,
                'type': 'Simbol dan Tanda Baca',
                'subtype': sym_name,
                'line': line_number
            }
            tokens['symbols'].append(token_info)
            all_tokens.append(token_info)
        elif kind == 'UNKNOWN':
            token_info = {
                'token': value,
                'type': 'Unknown',
                'subtype': 'UNKNOWN',
                'line': line_number
            }
            tokens['unknown'].append(token_info)
            all_tokens.append(token_info)

    return tokens, all_tokens

# ============================================================
# FUNGSI GENERATE HTML OUTPUT
# ============================================================

def get_color(token_type):
    colors = {
        'Reserve Word':         {'bg': '#1a3a5c', 'text': '#569cd6', 'badge': '#007acc'},
        'Simbol dan Tanda Baca': {'bg': '#2d2d2d', 'text': '#d4d4d4', 'badge': '#555'},
        'Variabel':             {'bg': '#1a3a4a', 'text': '#9cdcfe', 'badge': '#0e7490'},
        'Kalimat Matematika':   {'bg': '#1a3a1a', 'text': '#b5cea8', 'badge': '#166534'},
        'String Literal':       {'bg': '#3a2a1a', 'text': '#ce9178', 'badge': '#92400e'},
        'Komentar':             {'bg': '#1a2a1a', 'text': '#6a9955', 'badge': '#14532d'},
        'Unknown':              {'bg': '#3a1a1a', 'text': '#f44747', 'badge': '#991b1b'},
    }
    return colors.get(token_type, {'bg': '#2d2d2d', 'text': '#fff', 'badge': '#555'})

def generate_table_html(token_list, title="Token"):
    if not token_list:
        return f"""
        <div style='text-align:center; padding:30px; color:#666;
                    font-family:Consolas; background:#1e1e1e; border-radius:8px;'>
            <div style='font-size:40px;'>📭</div>
            <div>Tidak ada token ditemukan untuk kategori ini</div>
        </div>
        """

    rows_html = ""
    for i, t in enumerate(token_list, 1):
        c = get_color(t['type'])
        bg = '#1e1e1e' if i % 2 == 0 else '#252525'
        token_escaped = t['token'].replace('<','&lt;').replace('>','&gt;')
        rows_html += f"""
        <tr style='background:{bg};'>
            <td style='padding:8px 12px; color:#666; text-align:center;
                       font-size:12px;'>{i}</td>
            <td style='padding:8px 12px;'>
                <code style='background:{c['bg']}; color:{c['text']};
                             padding:2px 8px; border-radius:4px;
                             font-size:13px;'>{token_escaped}</code>
            </td>
            <td style='padding:8px 12px;'>
                <span style='background:{c['badge']}; color:white;
                             padding:2px 10px; border-radius:12px;
                             font-size:11px;'>{t['type']}</span>
            </td>
            <td style='padding:8px 12px; color:#888;
                       font-size:12px;'>{t['subtype']}</td>
            <td style='padding:8px 12px; color:#666;
                       text-align:center; font-size:12px;'>{t['line']}</td>
        </tr>
        """

    return f"""
    <div style='margin:10px 0;'>
        <div style='background:#252525; color:#aaa; padding:8px 16px;
                    font-size:12px; font-family:Consolas; border-radius:4px 4px 0 0;'>
            Total: <strong style='color:#fff;'>{len(token_list)}</strong> token
        </div>
        <div style='overflow-x:auto; border-radius:0 0 8px 8px;
                    border:1px solid #333;'>
            <table style='width:100%; border-collapse:collapse;
                          font-family:Consolas; font-size:13px;'>
                <thead>
                    <tr style='background:#2d2d2d;'>
                        <th style='padding:10px 12px; color:#888;
                                   text-align:center; width:50px;'>No</th>
                        <th style='padding:10px 12px; color:#ccc;
                                   text-align:left;'>Token</th>
                        <th style='padding:10px 12px; color:#ccc;
                                   text-align:left;'>Tipe</th>
                        <th style='padding:10px 12px; color:#ccc;
                                   text-align:left;'>Sub-Tipe</th>
                        <th style='padding:10px 12px; color:#888;
                                   text-align:center; width:70px;'>Baris</th>
                    </tr>
                </thead>
                <tbody>
                    {rows_html}
                </tbody>
            </table>
        </div>
    </div>
    """

def generate_stats_html(tokens, all_tokens):
    total = len(all_tokens)
    if total == 0:
        return "<p style='color:#666;'>Tidak ada data</p>"

    categories = [
        ('🔑 Reserve Words',        tokens['reserve_words'],     '#007acc'),
        ('⚙️ Simbol dan Tanda Baca', tokens['symbols'],           '#555'),
        ('📦 Variabel',              tokens['variables'],          '#0e7490'),
        ('🔢 Kalimat Matematika',    tokens['math_expressions'],  '#166534'),
        ('💬 String Literal',        tokens['strings'],            '#92400e'),
        ('📝 Komentar',              tokens['comments'],           '#14532d'),
        ('❓ Unknown',               tokens['unknown'],            '#991b1b'),
    ]

    bars_html = ""
    cards_html = ""

    for name, token_list, color in categories:
        count = len(token_list)
        pct = (count / total * 100) if total > 0 else 0
        bars_html += f"""
        <div style='margin:8px 0;'>
            <div style='display:flex; justify-content:space-between;
                        margin-bottom:4px;'>
                <span style='color:#ccc; font-size:13px;'>{name}</span>
                <span style='color:#888; font-size:12px;'>
                    {count} ({pct:.1f}%)
                </span>
            </div>
            <div style='background:#333; border-radius:10px;
                        height:10px; overflow:hidden;'>
                <div style='background:{color}; width:{pct}%;
                            height:100%; border-radius:10px;
                            transition:width 0.3s;'></div>
            </div>
        </div>
        """
        cards_html += f"""
        <div style='background:#252525; border:1px solid #333;
                    border-radius:8px; padding:15px; text-align:center;
                    min-width:130px; flex:1;'>
            <div style='font-size:24px;'>{name.split()[0]}</div>
            <div style='color:{color}; font-size:28px;
                        font-weight:bold; margin:5px 0;'>{count}</div>
            <div style='color:#666; font-size:11px;'>{name[2:]}</div>
            <div style='color:#888; font-size:11px;'>{pct:.1f}%</div>
        </div>
        """

    # Token frequency
    token_freq = {}
    for t in all_tokens:
        key = t['token']
        token_freq[key] = token_freq.get(key, 0) + 1

    sorted_tokens = sorted(token_freq.items(), key=lambda x: x[1], reverse=True)

    freq_rows = ""
    for i, (tok, freq) in enumerate(sorted_tokens[:10], 1):
        pct = freq / total * 100
        tok_escaped = tok.replace('<','&lt;').replace('>','&gt;')
        freq_rows += f"""
        <tr style='background:{"#1e1e1e" if i%2==0 else "#252525"};'>
            <td style='padding:6px 12px; color:#666;
                       text-align:center;'>{i}</td>
            <td style='padding:6px 12px;'>
                <code style='color:#9cdcfe;
                             background:#1a3a4a; padding:2px 8px;
                             border-radius:4px;'>{tok_escaped}</code>
            </td>
            <td style='padding:6px 12px; color:#fff;
                       text-align:center; font-weight:bold;'>{freq}</td>
            <td style='padding:6px 12px; color:#888;
                       text-align:center;'>{pct:.1f}%</td>
        </tr>
        """

    return f"""
    <!-- CARDS -->
    <div style='display:flex; gap:10px; flex-wrap:wrap; margin-bottom:20px;'>
        <div style='background:#1a3a5c; border:1px solid #007acc;
                    border-radius:8px; padding:15px; text-align:center;
                    min-width:130px; flex:1;'>
            <div style='font-size:24px;'>📊</div>
            <div style='color:#61dafb; font-size:32px;
                        font-weight:bold; margin:5px 0;'>{total}</div>
            <div style='color:#aaa; font-size:11px;'>Total Token</div>
        </div>
        {cards_html}
    </div>

    <!-- BAR CHART -->
    <div style='background:#1e1e1e; border-radius:8px;
                padding:20px; margin-bottom:20px;
                border:1px solid #333;'>
        <h3 style='color:#ccc; margin:0 0 15px 0;
                   font-size:14px;'>📈 Distribusi Token</h3>
        {bars_html}
    </div>

    <!-- FREQUENCY TABLE -->
    <div style='background:#1e1e1e; border-radius:8px;
                padding:20px; border:1px solid #333;'>
        <h3 style='color:#ccc; margin:0 0 15px 0;
                   font-size:14px;'>🔝 Top 10 Token Paling Sering Muncul</h3>
        <table style='width:100%; border-collapse:collapse;
                      font-family:Consolas; font-size:13px;'>
            <thead>
                <tr style='background:#2d2d2d;'>
                    <th style='padding:8px 12px; color:#888;
                               text-align:center;'>Rank</th>
                    <th style='padding:8px 12px; color:#ccc;
                               text-align:left;'>Token</th>
                    <th style='padding:8px 12px; color:#ccc;
                               text-align:center;'>Frekuensi</th>
                    <th style='padding:8px 12px; color:#ccc;
                               text-align:center;'>Persentase</th>
                </tr>
            </thead>
            <tbody>
                {freq_rows}
            </tbody>
        </table>
    </div>
    """

# ============================================================
# SAMPLE CODES
# ============================================================

SAMPLES = {
    'C++ Basic': '''// Program C++ Sederhana
#include <iostream>
using namespace std;

int main() {
    int x = 10;
    float pi = 3.14159;
    string name = "Hello World";

    if (x > 5 && pi < 4.0) {
        x += 1;
        pi *= 2;
    }

    for (int i = 0; i < x; i++) {
        cout << i << endl;
    }

    return 0;
}''',

    'Python': '''# Program Python
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

class MyClass:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hello, {self.name}"

x = 10
y = 3.14
result = x * y + 2 ** 8
print(result)

for i in range(10):
    if i % 2 == 0:
        print(i)''',

    'Java': '''// Program Java
import java.util.Scanner;

public class Main {
    public static void main(String[] args) {
        int x = 100;
        double y = 3.14;
        boolean flag = true;
        String message = "Hello Java";

        if (flag && x > 50) {
            x -= 10;
            y *= 2.0;
        } else {
            x++;
        }

        for (int i = 0; i <= 10; i++) {
            System.out.println(i * x);
        }
    }
}''',

    'Ekspresi Matematika': '''/* Contoh ekspresi matematika */
int a = 5 + 3 * 2;
float b = (10.5 - 3.2) / 2.0;
double c = a ** 2 + b ** 2;
bool check = (a >= 10) && (b <= 5.0);
int result = (a % 3 == 0) ? a : b;
x += 1;
y -= 2;
z *= 3;
w /= 4;
'''
}

# ============================================================
# UI DENGAN IPYWIDGETS
# ============================================================

def build_ui():
    # ---- CSS Style ----
    display(HTML("""
    <style>
        .widget-textarea textarea {
            background: #1e1e1e !important;
            color: #d4d4d4 !important;
            font-family: Consolas, monospace !important;
            font-size: 13px !important;
            border: 1px solid #444 !important;
            border-radius: 6px !important;
        }
        .widget-button button {
            border-radius: 6px !important;
            font-family: Consolas, monospace !important;
            font-weight: bold !important;
            border: none !important;
            cursor: pointer !important;
        }
        .widget-dropdown select {
            background: #2d2d2d !important;
            color: #fff !important;
            font-family: Consolas, monospace !important;
            border: 1px solid #444 !important;
            border-radius: 6px !important;
        }
    </style>
    """))

    # ---- HEADER ----
    display(HTML("""
    <div style='background: linear-gradient(135deg, #1e1e1e, #2d2d2d);
                border-radius: 12px; padding: 25px; margin-bottom: 15px;
                border: 1px solid #444; text-align: center;
                font-family: Consolas, monospace;'>
        <div style='font-size: 36px; margin-bottom: 5px;'>🔍</div>
        <h1 style='color: #61dafb; margin: 0; font-size: 26px;
                   letter-spacing: 2px;'>LEXICAL ANALYZER</h1>
        <p style='color: #888; margin: 8px 0 0 0; font-size: 14px;'>
            Tokenizer untuk Analisis Leksikal Program Komputer
        </p>
        <div style='margin-top: 12px;'>
            <span style='background: #1a3a5c; color: #569cd6; padding: 3px 10px;
                         border-radius: 12px; font-size: 11px; margin: 2px;
                         display: inline-block;'>🔑 Reserve Words</span>
            <span style='background: #2d2d2d; color: #d4d4d4; padding: 3px 10px;
                         border-radius: 12px; font-size: 11px; margin: 2px;
                         display: inline-block;'>⚙️ Simbol</span>
            <span style='background: #1a3a4a; color: #9cdcfe; padding: 3px 10px;
                         border-radius: 12px; font-size: 11px; margin: 2px;
                         display: inline-block;'>📦 Variabel</span>
            <span style='background: #1a3a1a; color: #b5cea8; padding: 3px 10px;
                         border-radius: 12px; font-size: 11px; margin: 2px;
                         display: inline-block;'>🔢 Matematika</span>
        </div>
    </div>
    """))

    # ---- INPUT AREA ----
    display(HTML("""
    <div style='color:#ccc; font-family:Consolas; font-size:13px;
                margin-bottom:6px;'>
        📝 <strong>Input Source Code:</strong>
        <span style='color:#666; font-size:11px; margin-left:10px;'>
            (Masukkan program yang ingin dianalisis)
        </span>
    </div>
    """))

    input_area = widgets.Textarea(
        value=SAMPLES['C++ Basic'],
        placeholder='Masukkan source code di sini...',
        layout=widgets.Layout(
            width='100%',
            height='250px'
        )
    )
    display(input_area)

    # ---- SAMPLE SELECTOR ----
    display(HTML("<div style='height:10px;'></div>"))

    sample_dropdown = widgets.Dropdown(
        options=list(SAMPLES.keys()),
        value='C++ Basic',
        description='📋 Sample:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    # ---- BUTTONS ----
    btn_analyze = widgets.Button(
        description='▶  ANALISIS TOKEN',
        button_style='',
        style={'button_color': '#007acc', 'font_weight': 'bold'},
        layout=widgets.Layout(width='200px', height='40px')
    )

    btn_clear = widgets.Button(
        description='🗑  CLEAR',
        button_style='danger',
        layout=widgets.Layout(width='130px', height='40px')
    )

    btn_load = widgets.Button(
        description='📂 LOAD SAMPLE',
        button_style='success',
        layout=widgets.Layout(width='170px', height='40px')
    )

    btn_row = widgets.HBox(
        [btn_analyze, btn_clear, btn_load, sample_dropdown],
        layout=widgets.Layout(gap='10px', align_items='center', margin='10px 0')
    )
    display(btn_row)

    # ---- OUTPUT AREA ----
    output = widgets.Output()
    display(output)

    # ---- EVENT HANDLERS ----

    def on_analyze(b):
        with output:
            clear_output(wait=True)
            src = input_area.value

            if not src.strip():
                display(HTML("""
                <div style='background:#3a1a1a; border:1px solid #991b1b;
                            border-radius:8px; padding:20px; text-align:center;
                            font-family:Consolas; color:#f44747;'>
                    ⚠️ Masukkan source code terlebih dahulu!
                </div>
                """))
                return

            try:
                tokens, all_tokens = tokenize(src)

                # Status bar
                total = len(all_tokens)
                display(HTML(f"""
                <div style='background:#1a3a1a; border:1px solid #166534;
                            border-radius:8px; padding:12px 20px;
                            font-family:Consolas; font-size:13px;
                            margin-bottom:15px; color:#b5cea8;'>
                    ✅ <strong style='color:#4ade80;'>Analisis Berhasil!</strong>
                    &nbsp;|&nbsp; Total: <strong>{total}</strong> token
                    &nbsp;|&nbsp; Reserve Words: <strong>{len(tokens['reserve_words'])}</strong>
                    &nbsp;|&nbsp; Simbol: <strong>{len(tokens['symbols'])}</strong>
                    &nbsp;|&nbsp; Variabel: <strong>{len(tokens['variables'])}</strong>
                    &nbsp;|&nbsp; Matematika: <strong>{len(tokens['math_expressions'])}</strong>
                </div>
                """))

                # TAB menggunakan Accordion
                tabs_data = [
                    ("📋 Semua Token",           all_tokens),
                    ("🔑 Reserve Words",          tokens['reserve_words']),
                    ("⚙️ Simbol & Tanda Baca",   tokens['symbols']),
                    ("📦 Variabel (Identifier)",  tokens['variables']),
                    ("🔢 Kalimat Matematika",     tokens['math_expressions']),
                    ("💬 String Literal",         tokens['strings']),
                    ("📝 Komentar",               tokens['comments']),
                ]

                accordion_children = []
                titles = []

                for title, token_list in tabs_data:
                    out = widgets.Output()
                    with out:
                        display(HTML(generate_table_html(token_list, title)))
                    accordion_children.append(out)
                    titles.append(f"{title} ({len(token_list)})")

                # Statistik
                stat_out = widgets.Output()
                with stat_out:
                    display(HTML(generate_stats_html(tokens, all_tokens)))
                accordion_children.append(stat_out)
                titles.append("📈 Statistik")

                accordion = widgets.Accordion(children=accordion_children)
                for i, title in enumerate(titles):
                    accordion.set_title(i, title)
                accordion.selected_index = 0

                display(accordion)

            except Exception as e:
                display(HTML(f"""
                <div style='background:#3a1a1a; border:1px solid #991b1b;
                            border-radius:8px; padding:20px;
                            font-family:Consolas; color:#f44747;'>
                    ❌ <strong>Error:</strong> {str(e)}
                </div>
                """))

    def on_clear(b):
        input_area.value = ''
        with output:
            clear_output()
            display(HTML("""
            <div style='background:#2d2d2d; border-radius:8px;
                        padding:20px; text-align:center;
                        font-family:Consolas; color:#666;'>
                🗑 Input dan output telah dibersihkan.
            </div>
            """))

    def on_load(b):
        selected = sample_dropdown.value
        input_area.value = SAMPLES[selected]
        with output:
            clear_output()
            display(HTML(f"""
            <div style='background:#1a3a1a; border-radius:8px;
                        padding:15px; font-family:Consolas;
                        color:#6a9955; border:1px solid #166534;'>
                ✅ Sample <strong style='color:#4ade80;'>"{selected}"</strong>
                berhasil dimuat. Klik <strong>"ANALISIS TOKEN"</strong> untuk memulai.
            </div>
            """))

    btn_analyze.on_click(on_analyze)
    btn_clear.on_click(on_clear)
    btn_load.on_click(on_load)

# ============================================================
# JALANKAN PROGRAM
# ============================================================
build_ui()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.6 MB/s eta 0:00:00


Textarea(value='// Program C++ Sederhana\n#include <iostream>\nusing namespace std;\n\nint main() {\n    int x…

Output()